In [ ]:
import pandas as pd

from statx_helpers import DATASET_SPECS, TARGET_COL, cross_val_by_period

input_files = {
    "all_drugs": "../outputs/train_all_drugs_merged.csv",
    "all_opioids": "../outputs/train_all_opioids_merged.csv",
    "all_stimulants": "../outputs/train_all_stimulants_merged.csv",
}


In [ ]:
results = []

# 5-fold GroupKFold by period (cross_val_by_period), one run per prediction
# type. Same engine as notebook 04 and 07 so numbers are directly comparable.
for prediction_type, path in input_files.items():
    print("\n============================================================")
    print(f"Prediction type: {prediction_type}")
    print("============================================================")

    raw_df = pd.read_csv(path)
    print("Rows:", len(raw_df.dropna(subset=[TARGET_COL])),
          "| Periods:", raw_df["period_id"].nunique(), "| Folds: 5")

    for dataset_name, spec in DATASET_SPECS.items():
        print(f"\n-------------------- {prediction_type} | {dataset_name} --------------------")
        results.extend(
            cross_val_by_period(
                raw_df,
                spec["method"],
                dataset_name=dataset_name,
                prediction_type=prediction_type,
                n_splits=5,
                drop_category=True,
            )
        )


In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values(by=["prediction_type", "RMSE"], ascending=[True, True])
    .reset_index(drop=True)
)

results_df


In [ ]:
results_df.to_csv(
    "../outputs/category_period_model_comparison.csv",
    index=False,
)

print("Saved:")
print("../outputs/category_period_model_comparison.csv")


In [ ]:
best_by_type = (
    results_df
    .sort_values("RMSE")
    .groupby("prediction_type")
    .head(1)
    .reset_index(drop=True)
)

best_by_type
